[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH06/06_02_end/notebooks/06_02_llm_judge.ipynb)

# 06_02 Evaluate with an LLM judge

Use a frontier model to score replies on qualities a rule checker can't measure, like grounding and how clear the next step is.

This notebook doesn't need a GPU. The default CPU runtime works fine. It does need an Anthropic API key. You don't need one for this course: watch this demo, then build your own judge in the next AI Practice.

In [ ]:
%%capture
!pip install anthropic

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH06")

import ridgeline

In [ ]:
import os
from google.colab import userdata

# Add your key in Colab Secrets as ANTHROPIC_API_KEY, with notebook access turned on.
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()
JUDGE_MODEL = "claude-sonnet-5-5"

## Load the replies to judge

In [ ]:
# Outputs from 06_01. Use the course copy, or change the path to your own file.
outputs = {o["id"]: o for o in ridgeline.load_tickets(ridgeline.DATA / "eval_outputs.jsonl")}
tickets = {t["id"]: t for t in ridgeline.load_tickets()}
print(f"Judging the {len(tickets)} hand-written seed tickets")

## Write the rubric

Four dimensions, each scored from 1 to 5. The judge gets the same rules and policies the assistant follows, and it has to reason before it scores.

In [ ]:
JUDGE_PROMPT = """You are grading a customer support reply for Ridgeline Outfitters.

Score the reply from 1 to 5 on each dimension:
- voice: sounds like the Ridgeline brand voice rules below. Warm, plain, specific, no banned phrases.
- structure: has the Category and Escalate lines, the greeting, the exact sign-off, and stays under 120 words.
- grounding: uses only facts from the ticket, the context, and the policies. Any invented detail scores 1.
- resolution: the customer knows exactly what happens next and when.

5 means fully meets the standard, 3 means noticeable problems, 1 means fails it.

Voice rules:
{voice_rules}

Policies:
{policies}

Ticket from {customer}:
{message}

What the support system knows: {context}

Reply to grade:
{reply}

Think through each dimension briefly, then end with only this JSON on the last line:
{{"voice": n, "structure": n, "grounding": n, "resolution": n}}"""


## Build the judge

In [ ]:
import json
import re

DIMENSIONS = ["voice", "structure", "grounding", "resolution"]


def judge(ticket, reply):
    prompt = JUDGE_PROMPT.format(
        voice_rules=ridgeline.voice_rules(), policies=ridgeline.policies(),
        customer=ticket["customer_name"], message=ticket["message"],
        context=ticket["context"], reply=reply,
    )
    response = client.messages.create(
        model=JUDGE_MODEL, max_tokens=2000,
        messages=[{"role": "user", "content": prompt}],
    )
    # The reply can start with a thinking block, so keep only the text blocks.
    text = "".join(block.text for block in response.content if block.type == "text")
    scores = json.loads(re.findall(r"\{[^{}]*\}", text)[-1])
    return {d: int(scores[d]) for d in DIMENSIONS}

## Judge both models

In [ ]:
rows = []
for ticket_id, ticket in tickets.items():
    for model_name in ["baseline", "finetuned"]:
        scores = judge(ticket, outputs[ticket_id][model_name])
        rows.append({"id": ticket_id, "model": model_name, **scores})
    print(f"  judged {ticket_id}")

## Compare the averages

A reply passes if it averages 4 or higher with no dimension below 3.

In [ ]:
import pandas as pd

scores = pd.DataFrame(rows)
scores["average"] = scores[DIMENSIONS].mean(axis=1)
scores["passes"] = (scores["average"] >= 4) & (scores[DIMENSIONS].min(axis=1) >= 3)
scores.groupby("model")[DIMENSIONS + ["average", "passes"]].mean().round(2)

## Read the lowest-scoring reply

Always read what the judge flags. Sometimes the reply is wrong, and sometimes the judge is.

In [ ]:
worst = scores[scores["model"] == "finetuned"].sort_values("average").iloc[0]
print(f"Lowest-scoring fine-tuned reply: {worst['id']}")
print(worst[DIMENSIONS].to_dict(), "\n")
print(outputs[worst["id"]]["finetuned"])